# AURA-RT — Colab Backend

Levanta el backend FastAPI en Google Colab con GPU, monta caché en Google Drive y lo expone via ngrok.

**Antes de ejecutar:**
1. Activar GPU: *Runtime → Change runtime type → T4 GPU*
2. Pegar `NGROK_AUTHTOKEN` en la **Celda 2**
3. Ejecutar todas las celdas en orden (*Runtime → Run all*)
4. Copiar la URL pública que aparece al final y pegarla en la web app

> ⚠️ Mantener la sesión de Colab activa mientras se use la plataforma.

## Celda 1 — Verificar GPU

In [ ]:
import subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(
        'GPU no disponible. Ir a Runtime → Change runtime type → T4 GPU y reiniciar.'
    )
print(f'GPU detectada: {result.stdout.strip()}')
print('✅ GPU OK')

## Celda 2 — Configuración

In [ ]:
import os

# REQUERIDO: pegar token de ngrok (https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTHTOKEN = 'PEGA_TU_TOKEN_AQUI'

# OPCIONAL
WEBAPP_URL = ''  # Dejar vacío si la web app está hosteada externamente

if NGROK_AUTHTOKEN == 'PEGA_TU_TOKEN_AQUI':
    raise RuntimeError('❌ Reemplazá PEGA_TU_TOKEN_AQUI con tu token real de ngrok.')

os.environ['NGROK_AUTHTOKEN']      = NGROK_AUTHTOKEN
os.environ['AURA_RT_PORT']         = '8000'
os.environ['AURA_RT_LOG_LEVEL']    = 'INFO'
os.environ['AURA_RT_CORS_ORIGINS'] = '*'
os.environ['AURA_RT_REPO_ROOT']    = '/content/AURAOnline'
if WEBAPP_URL:
    os.environ['AURA_RT_WEBAPP_URL'] = WEBAPP_URL

print('✅ Configuración lista.')

## Celda 3 — Clonar / actualizar repositorio

In [ ]:
import subprocess
from pathlib import Path

REPO_URL  = 'https://github.com/agusrosich/AURAOnline.git'
REPO_ROOT = Path('/content/AURAOnline')

if REPO_ROOT.exists():
    print('Repositorio ya presente — actualizando...')
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
else:
    print('Clonando repositorio...')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

rev = subprocess.check_output(
    ['git', '-C', str(REPO_ROOT), 'rev-parse', '--short', 'HEAD'], text=True
).strip()
print(f'✅ Repositorio listo — revisión: {rev}')

## Celda 4 — Google Drive + dependencias

El caché de pesos de TotalSegmentator (~2 GB) se guarda en Drive para no descargarlo en cada sesión.

In [ ]:
import os, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

CACHE_ROOT    = Path('/content/drive/MyDrive/AURA_RT_CACHE')
TOTALSEG_CACHE = CACHE_ROOT / 'totalsegmentator'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
TOTALSEG_CACHE.mkdir(parents=True, exist_ok=True)

os.environ['TOTALSEG_HOME_DIR']   = str(TOTALSEG_CACHE)
os.environ['AURA_RT_MODEL_CACHE'] = str(CACHE_ROOT)

print(f'Caché en: {CACHE_ROOT}')
print('Instalando dependencias (2-4 min la primera vez)...')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     '-r', '/content/AURAOnline/backend/requirements-colab.txt'],
    check=True
)
print('✅ Dependencias instaladas.')

## Celda 5 — Pre-descargar pesos de TotalSegmentator

Descarga los pesos antes de levantar el servidor para que el primer request no timeout.
Si los pesos ya están en Drive esta celda termina en segundos.

In [ ]:
import subprocess, sys

print('Verificando / descargando pesos de TotalSegmentator v2...')
print('(Primera vez: ~2 GB, puede tardar 5-10 min)')

# Importar el módulo fuerza la descarga de pesos sin procesar datos reales
result = subprocess.run(
    [sys.executable, '-c',
     'import totalsegmentator; print("TotalSegmentator", totalsegmentator.__version__)'],
    capture_output=True, text=True
)
print(result.stdout.strip() or 'TotalSegmentator importado')
if result.returncode != 0:
    print('STDERR:', result.stderr[:500])

check = subprocess.run(['TotalSegmentator', '--help'], capture_output=True, text=True)
print('TotalSegmentator en PATH:', check.returncode == 0)
print('✅ Listo.')

## Celda 6 — Levantar backend + ngrok

In [ ]:
import runpy

_ = runpy.run_path(
    '/content/AURAOnline/backend/scripts/colab_bootstrap.py',
    run_name='__main__'
)

## Celda 7 — Healthcheck y URL pública

Confirma que el backend responde correctamente y muestra la URL para copiar en la web app.

In [ ]:
import os, requests

public_url = os.environ.get('AURA_RT_PUBLIC_URL', '').strip()
port = os.environ.get('AURA_RT_PORT', '8000')

if not public_url:
    public_url = input('Pegá la URL pública de ngrok: ').strip()

headers = {'ngrok-skip-browser-warning': '1'}
endpoints = [
    ('local',   f'http://127.0.0.1:{port}/health'),
    ('publico', f'{public_url.rstrip("/")}/health'),
]

for label, url in endpoints:
    try:
        r = requests.get(url, timeout=15, headers=headers)
        data = r.json()
        n = data.get('supported_structure_count', '?')
        print(f'[{label}] HTTP {r.status_code} — {data.get("status")} — {n} estructuras')
    except Exception as e:
        print(f'[{label}] ERROR: {e}')

print()
print('=' * 60)
print('URL pública del backend:')
print(f'  {public_url}')
print('=' * 60)
print('Copiá esta URL en la web app → campo URL del servidor.')

## Celda 8 — Diagnóstico: nombres de ROI de TotalSegmentator

**Ejecutar solo si hay estructuras vacías en el RT-STRUCT.**

Segmenta un volumen sintético pequeño y lista los nombres de ROI reales generados,
para confirmar que coinciden con los esperados en `constants.py`.

In [ ]:
import subprocess, sys, tempfile
from pathlib import Path

GEN_VOLUME = '''
import numpy as np, SimpleITK as sitk, sys
from pathlib import Path
out = Path(sys.argv[1])
vol = np.zeros((32, 64, 64), dtype=np.int16)
vol[10:22, 20:44, 20:44] = -500
vol[5:8,   10:54, 10:54] = 700
img = sitk.GetImageFromArray(vol)
img.SetSpacing((2.0, 2.0, 3.0))
sitk.WriteImage(img, str(out))
print(f'Volumen: {img.GetSize()} spacing={img.GetSpacing()}')
'''

# Nombres de ROI esperados según constants.py
EXPECTED_ROIS = {
    'liver', 'spleen', 'kidney_right', 'kidney_left', 'pancreas',
    'gallbladder', 'aorta', 'stomach', 'small_bowel', 'colon',
    'urinary_bladder', 'prostate', 'brain', 'heart', 'trachea', 'esophagus',
    'lung_upper_lobe_right', 'lung_middle_lobe_right', 'lung_lower_lobe_right',
    'lung_upper_lobe_left', 'lung_lower_lobe_left',
    'hip_left', 'hip_right', 'sacrum',
    'femur_left', 'femur_right',
}

with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir   = Path(tmpdir)
    nifti_in = tmpdir / 'test.nii.gz'
    seg_out  = tmpdir / 'seg'

    subprocess.run([sys.executable, '-c', GEN_VOLUME, str(nifti_in)], check=True)
    print('Ejecutando TotalSegmentator --fast...')

    result = subprocess.run(
        ['TotalSegmentator', '-i', str(nifti_in), '-o', str(seg_out), '--fast'],
        capture_output=True, text=True
    )

    if result.returncode != 0:
        print('ERROR:', result.stderr[:1000])
    else:
        roi_files = sorted(f.stem.replace('.nii', '') for f in seg_out.glob('*.nii.gz'))
        print(f'TotalSegmentator generó {len(roi_files)} ROIs:')
        for roi in roi_files:
            print(f'  {roi}')

        missing = EXPECTED_ROIS - set(roi_files)
        extra   = set(roi_files) - EXPECTED_ROIS
        if missing:
            print(f'\n⚠️  Esperadas pero NO generadas ({len(missing)}):')
            for r in sorted(missing): print(f'  ✗ {r}')
        else:
            print('\n✅ Todos los nombres de ROI coinciden con constants.py')
        if extra:
            print(f'\nℹ️  Generadas pero no usadas por AURA-RT ({len(extra)}):')
            for r in sorted(extra): print(f'  + {r}')

## Celda 9 — Keep-alive (opcional)

Evita que Colab desconecte la sesión por inactividad. Interrumpir manualmente cuando termines.

In [ ]:
import time

print('Keep-alive activo. Interrumpir (■) cuando termines la sesión.')
i = 0
while True:
    i += 1
    print(f'[{i * 5} min] Backend activo...', flush=True)
    time.sleep(300)